# 05 — Generation

In [ ]:
import os
if not os.path.exists('MiniGPT'):
    !git clone https://github.com/userKk1/MiniGPT.git
%cd MiniGPT

Cloning into 'MiniGPT'...
remote: Enumerating objects: 105, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 105 (delta 46), reused 56 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (105/105), 163.98 KiB | 10.93 MiB/s, done.
Resolving deltas: 100% (46/46), done.
/content/MiniGPT


In [ ]:
import sys, json
sys.path.append('.')

import torch
from config import cfg, DATA_PROCESSED_DIR, CHECKPOINTS_DIR
from src.model import GPT

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

device: cuda


## 1. Load vocab and the best checkpoint

`ckpt_best.pt` (lowest val loss) is what we want here, not necessarily
the final training step's weights.

In [ ]:
!python -m src.data

README.md: 100% 1.30k/1.30k [00:00<00:00, 4.87MB/s]
Resolving data files: 100% 54/54 [00:00<00:00, 22806.61it/s]
Collected 4138 files, 20.00 MB
Wrote /content/MiniGPT/data/raw/corpus_raw.txt (21.0 MB)


In [ ]:
!python -m src.tokenizer

Corpus length: 20,964,485 characters
Vocab size: 99
train: 18,868,036 tokens
val:   2,096,449 tokens
Saved to /content/MiniGPT/data/processed


In [ ]:
!python -m src.train

device: cuda
vocab_size: 99  train tokens: 18,868,036  val tokens: 2,096,449
Parameters: 4,850,688
step     0 | train 4.6343 | val 4.6352 | lr 1.50e-06 | 9s
step   250 | train 2.5183 | val 2.5127 | lr 3.00e-04 | 47s
step   500 | train 2.1781 | val 2.1890 | lr 2.97e-04 | 88s
step   750 | train 1.9110 | val 1.9192 | lr 2.91e-04 | 131s
step  1000 | train 1.7171 | val 1.7275 | lr 2.82e-04 | 173s
step  1250 | train 1.5012 | val 1.5115 | lr 2.69e-04 | 216s
step  1500 | train 1.3841 | val 1.4001 | lr 2.54e-04 | 258s
step  1750 | train 1.3030 | val 1.3097 | lr 2.36e-04 | 300s
step  2000 | train 1.2436 | val 1.2576 | lr 2.17e-04 | 342s
step  2250 | train 1.1862 | val 1.2066 | lr 1.96e-04 | 384s
step  2500 | train 1.1571 | val 1.1703 | lr 1.74e-04 | 427s
step  2750 | train 1.1341 | val 1.1459 | lr 1.52e-04 | 470s
step  3000 | train 1.1027 | val 1.1228 | lr 1.30e-04 | 512s
step  3250 | train 1.1011 | val 1.1016 | lr 1.09e-04 | 555s
step  3500 | train 1.0654 | val 1.0975 | lr 9.00e-05 | 597s
step 

In [ ]:
tokenizer = Tokenizer.from_file(
    str(DATA_PROCESSED_DIR / "tokenizer.json")
)
vocab_size = tokenizer.get_vocab_size()

print(f"Vocabulary size: {vocab_size}")

model = GPT(vocab_size).to(device)
ckpt = torch.load(CHECKPOINTS_DIR / 'ckpt_best.pt', map_location=device)
model.load_state_dict(ckpt['model'])
model.eval()
print(f"Loaded checkpoint from step {ckpt['step']}, val_loss={ckpt['val_loss']:.4f}")

Loaded checkpoint from step 4999, val_loss=1.0323


## 2. Sampling strategies, briefly

- **Greedy** (temperature -> 0): always pick the single most likely next
  token. Deterministic, often repetitive/loops on itself.
- **Temperature**: scale logits before softmax. Low temperature (e.g. 0.3)
  sharpens the distribution (safer, more repetitive); high (e.g. 1.2) flattens
  it (more variety, more mistakes).
- **Top-k**: only sample from the k most likely tokens at each step — cuts
  off the long low-probability tail that produces nonsense.
- **Top-p (nucleus)**: like top-k, but the cutoff is dynamic — keep the
  smallest set of tokens whose cumulative probability exceeds p. Adapts to
  how confident the model is at each step, unlike top-k's fixed count.

In [ ]:
def generate_from_prompt(
    prompt,
    max_new_tokens=200,
    **kwargs
):
    encoded = tokenizer.encode(prompt)

    idx = torch.tensor(
        [encoded.ids],
        dtype=torch.long,
        device=device
    )

    # Generate new token IDs
    out = model.generate(
        idx,
        max_new_tokens=max_new_tokens,
        **kwargs
    )

    # Convert token IDs → Python code
    return tokenizer.decode(out[0].tolist())

## 3. Compare strategies on the same prompt

Using a real code prefix as the prompt — this is the actual 'code
completion' use case, closer to what you'd demo than generating from an
empty context.

In [ ]:
prompt = 'def factorial(n):\n    '

print('=== Greedy (temperature=0.01) ===')
print(generate_from_prompt(prompt, temperature=0.01))
print()

print('=== Temperature=0.8 ===')
print(generate_from_prompt(prompt, temperature=0.8))
print()

print('=== Top-k=40, temperature=0.8 ===')
print(generate_from_prompt(prompt, temperature=0.8, top_k=40))
print()

print('=== Top-p=0.9, temperature=0.8 ===')
print(generate_from_prompt(prompt, temperature=0.8, top_p=0.9))

=== Greedy (temperature=0.01) ===
def factorial(n):
                                                                                                                                                                                                            

=== Temperature=0.8 ===
def factorial(n):
                         config['src_manager'] = {'src_manager': 'src_manager'}
                                     {'test_help': 'src_manager'}
                                                          

=== Top-k=40, temperature=0.8 ===
def factorial(n):
        if n n == '':
               new_backend = new_backend
            self.add_event(element_backend, n)

    def get_check_event(self, context, return_vbackend):
         context.add_event(context, 

=== Top-p=0.9, temperature=0.8 ===
def factorial(n):
    """
    This program is set a data contained of the factory the specified of the specified
    a final defined by specified to containing defined to be the factory object.

    Th

## 4. Try a few more realistic completion prompts

Pick prompts similar to what a real code-completion demo would show —
partial function signatures, class definitions, import blocks.

In [ ]:
prompts = [
    'class Config:\n    def __init__(self):\n        ',
    'import numpy as np\n\ndef ',
    'def test_',
]

for p in prompts:
    print('PROMPT:', repr(p))
    print(generate_from_prompt(p, max_new_tokens=150, temperature=0.8, top_p=0.9))
    print('-' * 60)

PROMPT: 'class Config:\n    def __init__(self):\n        '
class Config:
    def __init__(self):
        return self.__init__(self.__init__, init__, max_label, max_label)

   def __init__(self, max_label):
        self.__init__(self, max_label)
    def __
------------------------------------------------------------
PROMPT: 'import numpy as np\n\ndef '
import numpy as np

def set_path_path(path_path_path_path, models, target_path):
    """Set path set path path path path path for path path path path paths paths. All is supp
------------------------------------------------------------
PROMPT: 'def test_'
def test_update(self, data, self):
        """Test the model all setting the default a set updated by a diction."""

        # If the model change or list
    
------------------------------------------------------------


## 5. Save a few samples for your results/

These go straight into `results/samples.md` — concrete evidence of what
the model produces, at a given checkpoint/step.

In [ ]:
from config import RESULTS_DIR

lines = [f"# Sample generations (checkpoint step {ckpt['step']}, val_loss={ckpt['val_loss']:.4f})", '']
for p in prompts:
    completion = generate_from_prompt(p, max_new_tokens=150, temperature=0.8, top_p=0.9)
    lines.append(f'## Prompt: `{p!r}`')
    lines.append('```python')
    lines.append(completion)
    lines.append('```')
    lines.append('')

(RESULTS_DIR / 'samples.md').write_text('\n'.join(lines))
print(f"Saved to {RESULTS_DIR / 'samples.md'}")

Saved to /content/MiniGPT/results/samples.md
